# Assembling the Full GPT Model Architecture

We've built the essential `Block`—the transformer's core computational unit. Now, it's time to zoom out and construct the full skyscraper. We will assemble the complete `GPT` model, integrating the input layers that give our model context and the output layer that allows it to make predictions.

By the end of this notebook, you will be able to build the full GPT model from its constituent parts. You will understand how token and positional embeddings work together to provide the initial context, and how a final "language model head" turns the transformer's deep processing into a concrete prediction for the next word. This is the final step in building the complete model architecture.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import graphviz

# --- Mock implementations from the previous notebook ---

class CausalSelfAttention(nn.Module):
    """ A simplified Causal Self-Attention module. """
    def __init__(self, config):
        super().__init__()
        self.n_head = config['n_head']
        self.n_embd = config['n_embd']
        # Key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(self.n_embd, 3 * self.n_embd)
        # Output projection
        self.c_proj = nn.Linear(self.n_embd, self.n_embd)
        # Causal mask
        self.register_buffer("bias", torch.tril(torch.ones(config['block_size'], config['block_size']))
                                     .view(1, 1, config['block_size'], config['block_size']))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality
        # Calculate query, key, values for all heads in batch
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        
        # Causal self-attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side-by-side
        # Output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    """ A simplified MLP module. """
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config['n_embd'], 4 * config['n_embd'])
        self.gelu    = nn.GELU(approximate='tanh')
        self.c_proj  = nn.Linear(4 * config['n_embd'], config['n_embd'])

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    """ A single Transformer Block. """
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config['n_embd'])
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config['n_embd'])
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

### The Core Idea: An Assembly Line for Meaning

Imagine our goal is to predict the next word in the sentence "The cat sat on the ___". The computer doesn't see words; it sees numbers, or "tokens." Let's say "The" is token `5`, "cat" is `12`, "sat" is `91`, etc. How does the model process this sequence of numbers to make a meaningful prediction?

Think of the `GPT` model as a sophisticated assembly line for building meaning.

1.  **The Parts Bin (Token Embeddings):** The first step is to grab the right "part" for each token. We have a massive parts bin, a lookup table, where every single word in our vocabulary has a corresponding high-dimensional vector—its "meaning." When the token `12` ("cat") comes in, we look up its vector. This is the **Token Embedding** layer (`wte`). It gives each word its initial identity.

2.  **The Conveyor Belt (Positional Embeddings):** Now we have the vectors for "The", "cat", "sat", "on", "the". But we've lost their order! "sat cat The" would have the same initial vectors. To solve this, we add a second vector to each one based on its position. The first word gets a "position 1" vector, the second gets a "position 2" vector, and so on. This is the **Positional Embedding** layer (`wpe`). By adding this to the token embedding, we create an input that knows both *what* a word is and *where* it is in the sequence.

3.  **The Assembly Stations (Transformer Blocks):** The combined embeddings then travel down a series of identical assembly stations. Each `Block` is a station that refines the meaning. It looks at all the words in the sequence (using self-attention) and updates each word's vector based on its context. After the first block, the vector for "sat" might have incorporated some meaning from "cat." After several blocks, its vector is richly contextualized.

4.  **The Final Inspection & Prediction (LM Head):** After passing through all the blocks, we have a final, highly-contextualized vector for each word. To predict what comes after "the", we take the *last* vector and pass it to a final linear layer, the **Language Model Head**. This layer's job is to take this rich meaning vector and project it back into a score for every single word in our vocabulary. The word with the highest score (hopefully "mat"!) is the model's prediction.

This entire pipeline—from token IDs to contextualized vectors to final word scores—is the `GPT` model.

In [ ]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        # The full "transformer" model is a dictionary of layers
        self.transformer = nn.ModuleDict(dict(
            # 1. Token Embedding: Maps token IDs to vectors
            wte = nn.Embedding(config['vocab_size'], config['n_embd']),
            # 2. Positional Embedding: Maps position indices to vectors
            wpe = nn.Embedding(config['block_size'], config['n_embd']),
            # 3. Transformer Blocks: A stack of processing layers
            h = nn.ModuleList([Block(config) for _ in range(config['n_layer'])]),
            # 4. Final LayerNorm: For stabilization before the output
            ln_f = nn.LayerNorm(config['n_embd']),
        ))
        # 5. Language Model Head: Maps final vectors to vocabulary scores
        self.lm_head = nn.Linear(config['n_embd'], config['vocab_size'], bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.size() # Batch size, Time (sequence length)
        
        # --- 1. Get initial token and position embeddings ---
        # Create position indices: 0, 1, 2, ..., T-1
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        # Look up token and position embeddings
        tok_emb = self.transformer.wte(idx)     # (B, T, n_embd)
        pos_emb = self.transformer.wpe(pos)     # (T, n_embd)
        
        # Add them together. Positional embeddings are broadcasted across the batch.
        x = tok_emb + pos_emb                   # (B, T, n_embd)

        # --- 2. Process through Transformer blocks ---
        for block in self.transformer.h:
            x = block(x)                        # (B, T, n_embd)

        # --- 3. Final normalization and language model head ---
        x = self.transformer.ln_f(x)            # (B, T, n_embd)
        logits = self.lm_head(x)                # (B, T, vocab_size)

        # --- 4. Calculate loss if targets are provided ---
        loss = None
        if targets is not None:
            # Reshape logits and targets for cross_entropy
            # Logits: from (B, T, V) to (B*T, V)
            # Targets: from (B, T) to (B*T)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

### Walking Through the Implementation

Let's break down the `GPT` class piece by piece.

**`__init__(self, config)`: Building the Assembly Line**

This is where we define and initialize all the layers (the "stations") of our model.
-   `wte = nn.Embedding(config['vocab_size'], config['n_embd'])`: This is our parts bin. It's a simple lookup table. If our vocabulary has 50,000 words (`vocab_size`) and our embedding dimension is 768 (`n_embd`), this layer is a `[50000, 768]` matrix.
-   `wpe = nn.Embedding(config['block_size'], config['n_embd'])`: This is the conveyor belt station that adds position info. If our model can handle sequences up to 1024 tokens long (`block_size`), this is a `[1024, 768]` matrix, one vector for each possible position.
-   `h = nn.ModuleList([Block(config) for _ in range(config['n_layer'])])`: This creates our stack of assembly stations. If `n_layer` is 12, this creates 12 `Block` modules and holds them in a list.
-   `ln_f = nn.LayerNorm(config['n_embd'])`: The final quality control station, a standard LayerNorm.
-   `self.lm_head = nn.Linear(...)`: The final prediction station. It's a linear layer that projects the final `n_embd`-dimensional vector into a `vocab_size`-dimensional vector of scores (logits).

**`forward(self, idx, targets=None)`: Running a Batch Through the Line**

This method defines the flow of data through the model.
-   `pos = torch.arange(0, T, ...)`: We first create the position indices `[0, 1, 2, ..., T-1]` for the input sequence.
-   `tok_emb = self.transformer.wte(idx)`: We look up the token embeddings for our input `idx`. If `idx` has shape `[8, 64]` (batch of 8, sequence of 64), `tok_emb` will have shape `[8, 64, 768]`.
-   `pos_emb = self.transformer.wpe(pos)`: We look up the positional embeddings. This returns a shape of `[64, 768]`.
-   `x = tok_emb + pos_emb`: Here's the magic. We add the two embeddings. PyTorch's broadcasting automatically adds the `pos_emb` of shape `[64, 768]` to every sequence in the batch of `tok_emb` (`[8, 64, 768]`), resulting in the final input `x` of shape `[8, 64, 768]`.
-   `for block in self.transformer.h: x = block(x)`: We simply loop through our `Block` modules, feeding the output of one as the input to the next. The shape remains `[8, 64, 768]`.
-   `logits = self.lm_head(x)`: We pass the final processed vectors through the LM head. This changes the shape from `[8, 64, 768]` to `[8, 64, 50000]`, giving us a score for every possible next word at every position in the sequence.
-   `loss = F.cross_entropy(...)`: If we are training, we have `targets` (the correct next words). We calculate the cross-entropy loss between our predicted `logits` and the `targets` to see how well the model is doing.

In [ ]:
# --- Demonstration ---

# Create a mock configuration
config = {
    'block_size': 128,    # Maximum sequence length
    'vocab_size': 500,     # Number of tokens in our vocabulary
    'n_layer': 4,         # Number of transformer blocks
    'n_head': 4,          # Number of attention heads
    'n_embd': 64,         # Embedding dimension
}

# Create the model
gpt_model = GPT(config)
print(f"GPT model created with {config['n_layer']} layers.")

# Create some dummy input data
batch_size = 8
seq_len = 32 # Can be shorter than block_size

# Input token indices
# A batch of 8 sequences, each 32 tokens long
# The token IDs are random integers between 0 and vocab_size-1
dummy_indices = torch.randint(0, config['vocab_size'], (batch_size, seq_len))

# Target token indices (the next word in the sequence)
dummy_targets = torch.randint(0, config['vocab_size'], (batch_size, seq_len))

print(f"Input indices shape: {dummy_indices.shape}")

# Perform a forward pass
logits, loss = gpt_model(dummy_indices, dummy_targets)

print(f"Output logits shape: {logits.shape}")
print(f"Expected logits shape: {(batch_size, seq_len, config['vocab_size'])}")
print(f"Calculated loss: {loss.item():.4f}")

# The loss should be around -ln(1/vocab_size), since the model is randomly initialized
# For vocab_size=500, this is -ln(1/500) = ln(500) approx 6.21
print(f"Expected initial loss (approx): {math.log(config['vocab_size']):.4f}")

### Going Deeper: Weight Tying

In the official nanoGPT repository, you'll find this curious line of code in the `GPT` model's `__init__` method:

```python
self.transformer.wte.weight = self.lm_head.weight
```

This is a technique called **weight tying**. Let's break down why this is a clever optimization.

-   The **token embedding layer** (`wte`) is essentially a matrix of size `[vocab_size, n_embd]`. Its job is to map a token ID (an integer) to a dense vector representation. It answers the question: "What does the token 'cat' *mean* in our vector space?"

-   The **language model head** (`lm_head`) is a linear layer with a weight matrix of size `[n_embd, vocab_size]`. Its job is to take a final, contextualized vector and map it back to a score for every token in the vocabulary. It answers the question: "Which token's meaning is *closest* to this final computed vector?"

These two operations are conceptually inverses of each other. One maps from the discrete world of tokens to the continuous world of vectors; the other maps back. The insight behind weight tying is that they should share the *same mapping*. We force the weight matrix of the output layer to be the transpose of the input embedding matrix.

**Analogy:** Imagine a bilingual dictionary. The `wte` is like looking up an English word to find its French translation. The `lm_head` is like having a French translation and looking for the original English word. It makes perfect sense to use the *same book* for both operations, just reading it in different directions.

**Why do this?**
1.  **Reduces Parameters:** The embedding and final layer are often two of the largest parameter sets in the model. Tying them drastically reduces the total number of parameters, making the model smaller and faster to train.
2.  **Improves Performance:** It acts as a form of regularization. It enforces a sensible symmetry on the model, preventing it from learning two separate, potentially misaligned mappings for the same core task. This often leads to better and more stable training.

This simple one-line change is an elegant and effective trick used in many modern transformer architectures.

In [ ]:
# --- Visualization of the GPT Architecture ---

def visualize_gpt_architecture(config):
    dot = graphviz.Digraph('GPT_Architecture', comment='The GPT Model')
    dot.attr('node', shape='box', style='rounded', color='lightblue2')
    dot.attr(rankdir='TB', splines='ortho')

    # Main graph
    with dot.subgraph(name='cluster_main') as c:
        c.attr(label='GPT Model', style='rounded')
        
        # Inputs
        c.node('idx', 'Input Tokens\n(B, T)')
        c.node('pos', 'Position Indices\n(T)')
        
        # Embeddings
        c.node('wte', 'Token Embedding (wte)\n(B, T, C)')
        c.node('wpe', 'Positional Embedding (wpe)\n(T, C)')
        c.node('add', '+', shape='circle', style='filled', color='lightgrey')

        c.edge('idx', 'wte')
        c.edge('pos', 'wpe')
        c.edge('wte', 'add')
        c.edge('wpe', 'add')
        
        # Transformer Blocks
        with c.subgraph(name='cluster_blocks') as b:
            b.attr(label=f"{config['n_layer']} x Transformer Blocks", style='rounded', color='lightgrey')
            b.node('block_1', 'Block 1')
            b.node('block_dots', '...', shape='plaintext')
            b.node('block_n', f"Block {config['n_layer']}")
            b.edge('block_1', 'block_dots', style='invis')
            b.edge('block_dots', 'block_n', style='invis')

        # Final Layers
        c.node('ln_f', 'Final LayerNorm\n(B, T, C)')
        c.node('lm_head', 'LM Head (Linear)\n(B, T, vocab_size)')
        c.node('logits', 'Output Logits\n(B, T, vocab_size)')

        # Connections
        c.edge('add', 'block_1')
        c.edge('block_n', 'ln_f')
        c.edge('ln_f', 'lm_head')
        c.edge('lm_head', 'logits')

    # Add annotation for weight tying
    dot.edge('wte', 'lm_head', style='dashed', constraint='false', color='crimson',
             label=' Weight Tying', fontcolor='crimson')
    
    return dot

# Use the same config from our demonstration
viz_config = {
    'n_layer': 4,
}
# Generate and display the graph
graph = visualize_gpt_architecture(viz_config)
graph

### Summary

In this notebook, we assembled the final `GPT` model, bringing together all the components we've discussed. We saw how the model ingests simple token IDs and, through a series of carefully designed steps, produces predictions for the next token in a sequence.

**What We Built:**
- A complete, functional `GPT` class that stacks multiple Transformer `Block`s.
- An input stage that combines `TokenEmbeddings` (what a word is) and `PositionalEmbeddings` (where it is).
- An output stage, the `lm_head`, that converts the model's internal representation back into scores over the entire vocabulary.

**Key Takeaways:**
- The overall GPT architecture is surprisingly straightforward: embed, process, and predict.
- The addition of positional information to token embeddings is a non-negotiable step for processing sequential data. Without it, the model is just a "bag of words."
- The `forward` pass shows the logical flow of data, from raw indices to final logits, ready to be compared against target labels for calculating loss.
- Techniques like weight tying are elegant optimizations that reduce model size and can improve performance by enforcing a logical symmetry in the architecture.

**What's Next?**
We now have a complete, working model architecture. But a model is useless without data. In the next notebook, `Preparing Text Data for GPT Training`, we will shift our focus from the model to the data itself. We'll learn how to take a large corpus of raw text, tokenize it, and create the batches of `inputs` and `targets` needed to train our GPT model efficiently.